In [1]:
import pandas as pd
from datetime import datetime as dt
from datetime import date

from kiblib.utils.db import DbConn

In [12]:
db_conn = DbConn().create_engine()

In [39]:
query = """SELECT i.itemnumber, i.biblionumber, i.biblioitemnumber, i.barcode,
i.dateaccessioned, i.homebranch, i.notforloan, i.itemcallnumber, i.location, i.ccode, bi.itemtype, b.title
FROM koha_prod.items i
JOIN koha_prod.biblioitems bi ON bi.biblionumber = i.biblionumber
JOIN koha_prod.biblio b ON b.biblionumber = i.biblionumber
WHERE bi.itemtype = 'PE' AND i.notforloan = '0'"""

items = pd.read_sql(query,db_conn)
len(items)

5659

In [40]:
perios = items.groupby(['biblionumber', 'title', 'homebranch', 'ccode'])['itemnumber'].count().reset_index()

In [41]:
query = """SELECT biblionumber, itemnumber, issue_id, borrowernumber, branch
FROM statdb.stat_issues
WHERE itemtype = 'PE'
AND DATE(issuedate) >= CURDATE() - INTERVAL 1 YEAR"""

prets = pd.read_sql(query,db_conn)
len(prets)

17413

In [42]:
prets_titre_nb = prets.groupby(['biblionumber', 'branch'])['issue_id'].count().reset_index()
prets_titre_emprunteurs_distincts = prets.groupby(['biblionumber', 'branch'])['borrowernumber'].nunique().reset_index()

In [43]:
perios = perios.merge(prets_titre_nb, left_on=['biblionumber', 'homebranch'], right_on=['biblionumber', 'branch'], how='left')
perios = perios[['biblionumber', 'title', 'homebranch', 'ccode', 'itemnumber', 'issue_id']]

In [44]:
perios = perios.merge(prets_titre_emprunteurs_distincts, left_on=['biblionumber', 'homebranch'], right_on=['biblionumber', 'branch'], how='left')
perios = perios[['biblionumber', 'title', 'homebranch', 'ccode', 'itemnumber', 'issue_id', 'borrowernumber']]

In [45]:
query = """SELECT av.authorised_value,av.lib 
FROM koha_prod.authorised_values av
WHERE category = 'collection'"""
va_collection = pd.read_sql(query, db_conn)

In [46]:
#perios = perios[perios['itemnumber'] > 1]
perios = perios.merge(va_collection, left_on='ccode', right_on='authorised_value', how='left')
perios

,biblionumber,title,homebranch,ccode,itemnumber,issue_id,borrowernumber,authorised_value,lib
0,154602,Spirou,MED,JPRZZZZ,58,109.0,19.0,NaN,NaN
1,154603,Alternatives économiques,MED,ACFPAZZ,19,65.0,34.0,ACFPAZZ,ACF - Presse d'actualité
2,154605,Archéologia,MED,ACFHSZZ,17,8.0,8.0,ACFHSZZ,ACF - Histoire
3,154607,Astrapi,BUS,JPRZZZZ,23,125.0,28.0,NaN,NaN
4,154607,Astrapi,MED,JPRZZZZ,46,260.0,65.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...
433,376436,Politis,MED,P17,18,2.0,2.0,P17,P17 - PERIODIQUE
434,376437,Futu&r,MED,P17,1,1.0,1.0,P17,P17 - PERIODIQUE
435,376438,L'écologiste,MED,P17,1,1.0,1.0,P17,P17 - PERIODIQUE
436,376439,Plénior,MED,P17,2,2.0,2.0,P17,P17 - PERIODIQUE


In [47]:
perios = perios[['biblionumber', 'title', 'ccode', 'lib', 'homebranch', 'itemnumber',
       'issue_id', 'borrowernumber']]

In [48]:
perios.columns = ['notice', 'titre', 'ccode', 'collection', 'site', 'nb exemplaires',
       'nb prêts', 'emprunteurs distincts']

In [49]:
perios['nb prêts'] = perios['nb prêts'].fillna(0)
perios['nb prêts'] = perios['nb prêts'].astype(int)

perios['emprunteurs distincts'] = perios['emprunteurs distincts'].fillna(0)
perios['emprunteurs distincts'] = perios['emprunteurs distincts'].astype(int)

/tmp/ipykernel_18202/753565272.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  perios['nb prêts'] = perios['nb prêts'].fillna(0)
/tmp/ipykernel_18202/753565272.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  perios['nb prêts'] = perios['nb prêts'].astype(int)
/tmp/ipykernel_18202/753565272.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydat

In [51]:
perios.loc[perios['ccode'] == 'JPRZZZZ', 'collection'] = 'Jeunesse - presse'
perios.loc[perios['ccode'] == 'AMVZZ', 'collection'] = 'AMV - Musique Généralités'
perios = perios.sort_values(by='ccode')
perios = perios[['notice', 'titre', 'collection', 'site', 'nb exemplaires',
       'nb prêts', 'emprunteurs distincts']]

In [52]:
query = """SELECT biblionumber, itemnumber, issue_id, borrowernumber, branch
FROM statdb.stat_issues
WHERE itemtype = 'PE'
AND DATE(issuedate) >= CURDATE() - INTERVAL 2 YEAR
AND DATE(issuedate) <= CURDATE() - INTERVAL 1 YEAR"""

prets2 = pd.read_sql(query,db_conn)
len(prets2)

18624

In [53]:
prets_titre_nb2 = prets2.groupby(['biblionumber', 'branch'])['issue_id'].count().reset_index()
prets_titre_emprunteurs_distincts2 = prets2.groupby(['biblionumber', 'branch'])['borrowernumber'].nunique().reset_index()

In [54]:
#perios.columns = ['notice', 'titre', 'auteur', 'ccode', 'collection', 'site', 'nb exemplaires',
#       'nb prêts', 'emprunteurs distincts']

prets_titre_nb2.columns = ['notice', 'site', 'nb prêts n-1']
prets_titre_emprunteurs_distincts2.columns = ['notice', 'site', 'emprunteurs distincts n-1']
perios = perios.merge(prets_titre_nb2, left_on=['notice', 'site'], right_on=['notice', 'site'], how='left')
perios = perios.merge(prets_titre_emprunteurs_distincts2, left_on=['notice', 'site'], right_on=['notice', 'site'], how='left')
perios

,notice,titre,collection,site,nb exemplaires,nb prêts,emprunteurs distincts,nb prêts n-1,emprunteurs distincts n-1
0,266342,Graffiti art,AAP - Arts plastiques,MED,7,18,10,14.0,13.0
1,156766,Art press,AAP - Arts plastiques,MED,12,22,10,25.0,9.0
2,230964,Polka,AAP - Arts plastiques,MED,4,15,10,14.0,6.0
3,154610,Beaux-Arts magazine,AAP - Arts plastiques,MED,12,58,30,54.0,28.0
4,266902,Home magazine,AAP - Arts plastiques,MED,9,56,21,48.0,26.0
...,...,...,...,...,...,...,...,...,...
433,156470,La Croix du Nord,PRR - Région,MED,25,1,1,NaN,NaN
434,235028,Eco 121,PRR - Région,MED,26,0,0,NaN,NaN
435,154840,Nord généalogie,PRR - Région,MED,10,1,1,NaN,NaN
436,154806,Le Carnet et les instants,PRR - Région,MED,12,0,0,9.0,2.0


In [55]:
perios['nb prêts n-1'] = perios['nb prêts n-1'].fillna(0)
perios['nb prêts n-1'] = perios['nb prêts n-1'].astype(int)

perios['emprunteurs distincts n-1'] = perios['emprunteurs distincts n-1'].fillna(0)
perios['emprunteurs distincts n-1'] = perios['emprunteurs distincts n-1'].astype(int)
perios

,notice,titre,collection,site,nb exemplaires,nb prêts,emprunteurs distincts,nb prêts n-1,emprunteurs distincts n-1
0,266342,Graffiti art,AAP - Arts plastiques,MED,7,18,10,14,13
1,156766,Art press,AAP - Arts plastiques,MED,12,22,10,25,9
2,230964,Polka,AAP - Arts plastiques,MED,4,15,10,14,6
3,154610,Beaux-Arts magazine,AAP - Arts plastiques,MED,12,58,30,54,28
4,266902,Home magazine,AAP - Arts plastiques,MED,9,56,21,48,26
...,...,...,...,...,...,...,...,...,...
433,156470,La Croix du Nord,PRR - Région,MED,25,1,1,0,0
434,235028,Eco 121,PRR - Région,MED,26,0,0,0,0
435,154840,Nord généalogie,PRR - Région,MED,10,1,1,0,0
436,154806,Le Carnet et les instants,PRR - Région,MED,12,0,0,9,2


In [56]:
perios['évolution prêts'] = round( (perios['nb prêts'] - perios['nb prêts n-1']) / perios['nb prêts n-1'] * 100 ,1)
perios['évolution emprunteurs distincts'] = round( (perios['emprunteurs distincts'] - perios['emprunteurs distincts n-1']) / perios['emprunteurs distincts n-1'] * 100,1)

In [57]:
perios

,notice,titre,collection,site,nb exemplaires,nb prêts,emprunteurs distincts,nb prêts n-1,emprunteurs distincts n-1,évolution prêts,évolution emprunteurs distincts
0,266342,Graffiti art,AAP - Arts plastiques,MED,7,18,10,14,13,28.6,-23.1
1,156766,Art press,AAP - Arts plastiques,MED,12,22,10,25,9,-12.0,11.1
2,230964,Polka,AAP - Arts plastiques,MED,4,15,10,14,6,7.1,66.7
3,154610,Beaux-Arts magazine,AAP - Arts plastiques,MED,12,58,30,54,28,7.4,7.1
4,266902,Home magazine,AAP - Arts plastiques,MED,9,56,21,48,26,16.7,-19.2
...,...,...,...,...,...,...,...,...,...,...,...
433,156470,La Croix du Nord,PRR - Région,MED,25,1,1,0,0,inf,inf
434,235028,Eco 121,PRR - Région,MED,26,0,0,0,0,NaN,NaN
435,154840,Nord généalogie,PRR - Région,MED,10,1,1,0,0,inf,inf
436,154806,Le Carnet et les instants,PRR - Région,MED,12,0,0,9,2,-100.0,-100.0


In [58]:
perios.to_excel("perios.xlsx", index=False)